In [1]:
import os
import requests
from getpass import getpass
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [2]:
@tool
def calculator(a: float, b: float, operation: str) -> str:
    """Calculate two numbers using add, subtract, multiply or divide."""

    if operation == "add":
        return str(a + b)
    elif operation == "subtract":
        return str(a - b)
    elif operation == "multiply":
        return str(a * b)
    elif operation == "divide":
        if b == 0:
            return "Error: Cannot divide by zero"
        return str(a / b)
    else:
        return "Unsupported operation"

In [3]:
calculator.invoke({
    "a": 25,
    "b": 12,
    "operation": "multiply"
})

'300.0'

In [4]:


@tool
def check_weather(city: str) -> str:
    """Get current weather and temperature for a city."""

    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "en"},
            timeout=15
        )
        geo.raise_for_status()
        places = geo.json().get("results", [])

        if not places:
            return f"Could not find city: {city}"

        place = places[0]

        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,relative_humidity_2m,wind_speed_10m"
            },
            timeout=15
        )
        response.raise_for_status()
        weather = response.json()["current"]

        return (
            f"Weather in {place['name']}: "
            f"{weather['temperature_2m']}°C, "
            f"humidity {weather['relative_humidity_2m']}%, "
            f"wind {weather['wind_speed_10m']} km/h."
        )

    except requests.RequestException as exc:
        return f"Weather service error: {exc}"
    except (KeyError, IndexError, ValueError) as exc:
        return f"Could not interpret weather data: {exc}"

In [9]:
check_weather.invoke({"city": "Hyderabad"})

'Weather in Hyderabad: 28.9°C, humidity 48%, wind 6.5 km/h.'

In [5]:

tools = [calculator, check_weather]

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful personal assistant. "
        "Use the calculator for arithmetic. "
        "Use check_weather for current weather questions. "
        "Never invent tool results."
    )
)

In [7]:

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Calculate 25 multiplied by 12, "
                "and check the current weather in Karimnagar."
            )
        }
    ]
})

print(response["messages"][-1].content)

- 25 × 12 = **300**  
- Current weather in Karimnagar: **27.1 °C**, humidity **71 %**, wind **6 km/h**.


In [8]:
user = input("input message")
response = agent.invoke({
    "messages": user
    })
print(response["messages"][-1].content)

The result of subtracting 2 from 1 is **‑1**.


In [10]:

response = agent.invoke({
    "messages": "weather in hyd"
    })
print(response["messages"][-1].content)

Here’s the current weather for Hyderabad:

- **Temperature:** 25.8 °C  
- **Humidity:** 74 %  
- **Wind:** 5.6 km/h  

Let me know if you’d like a forecast, details for another location, or anything else!


In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                
                "Check the current weather in Karimnagar and warangal."
            )
        }
    ]
})

print(response["messages"][-1].content)

**Current Weather**

- **Karimnagar:** 28.0 °C, humidity 63 %, wind 3.3 km/h  
- **Warangal:** 27.9 °C, humidity 73 %, wind 11.3 km/h
